In [3]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.ensemble import IsolationForest
from sklearn.feature_selection import mutual_info_classif


In [9]:
# Load the dataset
data = pd.read_csv("adult_with_headers.csv")

In [5]:
# Basic data exploration
print(data.head())

   age          workclass  fnlwgt   education  education_num  \
0   39          State-gov   77516   Bachelors             13   
1   50   Self-emp-not-inc   83311   Bachelors             13   
2   38            Private  215646     HS-grad              9   
3   53            Private  234721        11th              7   
4   28            Private  338409   Bachelors             13   

        marital_status          occupation    relationship    race      sex  \
0        Never-married        Adm-clerical   Not-in-family   White     Male   
1   Married-civ-spouse     Exec-managerial         Husband   White     Male   
2             Divorced   Handlers-cleaners   Not-in-family   White     Male   
3   Married-civ-spouse   Handlers-cleaners         Husband   Black     Male   
4   Married-civ-spouse      Prof-specialty            Wife   Black   Female   

   capital_gain  capital_loss  hours_per_week  native_country  income  
0          2174             0              40   United-States   <=50

In [7]:
print(data.describe())

                age        fnlwgt  education_num  capital_gain  capital_loss  \
count  32561.000000  3.256100e+04   32561.000000  32561.000000  32561.000000   
mean      38.581647  1.897784e+05      10.080679   1077.648844     87.303830   
std       13.640433  1.055500e+05       2.572720   7385.292085    402.960219   
min       17.000000  1.228500e+04       1.000000      0.000000      0.000000   
25%       28.000000  1.178270e+05       9.000000      0.000000      0.000000   
50%       37.000000  1.783560e+05      10.000000      0.000000      0.000000   
75%       48.000000  2.370510e+05      12.000000      0.000000      0.000000   
max       90.000000  1.484705e+06      16.000000  99999.000000   4356.000000   

       hours_per_week  
count    32561.000000  
mean        40.437456  
std         12.347429  
min          1.000000  
25%         40.000000  
50%         40.000000  
75%         45.000000  
max         99.000000  


In [9]:
print(data.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32561 entries, 0 to 32560
Data columns (total 15 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   age             32561 non-null  int64 
 1   workclass       32561 non-null  object
 2   fnlwgt          32561 non-null  int64 
 3   education       32561 non-null  object
 4   education_num   32561 non-null  int64 
 5   marital_status  32561 non-null  object
 6   occupation      32561 non-null  object
 7   relationship    32561 non-null  object
 8   race            32561 non-null  object
 9   sex             32561 non-null  object
 10  capital_gain    32561 non-null  int64 
 11  capital_loss    32561 non-null  int64 
 12  hours_per_week  32561 non-null  int64 
 13  native_country  32561 non-null  object
 14  income          32561 non-null  object
dtypes: int64(6), object(9)
memory usage: 3.7+ MB
None


In [11]:
# Handling missing values
data.dropna(inplace=True)  # Drop rows with missing values

In [15]:
# Scaling techniques
scaler_standard = StandardScaler()
scaler_minmax = MinMaxScaler()


In [17]:
# Assuming 'age' and 'education-num' are numerical features
data['age_scaled_standard'] = scaler_standard.fit_transform(data[['age']])
data['age_scaled_minmax'] = scaler_minmax.fit_transform(data[['age']])

Task 2: Encoding Techniques

In [19]:
from sklearn.preprocessing import OneHotEncoder, LabelEncoder

In [21]:
# One-Hot Encoding
onehot_cols = ['workclass', 'marital_status', 'occupation', 'relationship', 'race']
for col in onehot_cols:
    if len(data[col].unique()) < 5:
        onehot_encoder = OneHotEncoder(drop='first')
        encoded_cols = pd.DataFrame(onehot_encoder.fit_transform(data[[col]]).toarray(),
                                    columns=[col + '_' + str(i) for i in range(1, len(data[col].unique()))])
        data = pd.concat([data, encoded_cols], axis=1)

In [23]:
onehot_cols

['workclass', 'marital_status', 'occupation', 'relationship', 'race']

In [25]:
data

,age,workclass,fnlwgt,education,education_num,marital_status,occupation,relationship,race,sex,capital_gain,capital_loss,hours_per_week,native_country,income,age_scaled_standard,age_scaled_minmax
0,39,State-gov,77516,Bachelors,13,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40,United-States,<=50K,0.030671,0.301370
1,50,Self-emp-not-inc,83311,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,13,United-States,<=50K,0.837109,0.452055
2,38,Private,215646,HS-grad,9,Divorced,Handlers-cleaners,Not-in-family,White,Male,0,0,40,United-States,<=50K,-0.042642,0.287671
3,53,Private,234721,11th,7,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,0,0,40,United-States,<=50K,1.057047,0.493151
4,28,Private,338409,Bachelors,13,Married-civ-spouse,Prof-specialty,Wife,Black,Female,0,0,40,Cuba,<=50K,-0.775768,0.150685
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
32556,27,Private,257302,Assoc-acdm,12,Married-civ-spouse,Tech-support,Wife,White,Female,0,0,38,United-States,<=50K,-0.849080,0.136986
32557,40,Private,154374,HS-grad,9,Married-civ-spouse,Machine-op-inspct,Husband,White,Male,0,0,40,United-States,>50K,0.103983,0.315068
32558,58,Private,151910,HS-grad,9,Widowed,Adm-clerical,Unmarried,White,Female,0,0,40,United-States,<=50K,1.423610,0.561644
32559,22,Private,201490,HS-grad,9,Never-married,Adm-clerical,Own-child,White,Male,0,0,20,United-States,<=50K,-1.215643,0.068493


Task 3: Feature Engineering

In [21]:
# Display basic information about the dataset
print("Dataset Information:")
data.info()

# Feature 1: Work Hours Category
def work_hours_category(hours):
    if hours < 20:
        return 'Part-time'
    elif hours <= 40:
        return 'Full-time'
    else:
        return 'Overtime'

data['Work_Hours_Category'] = df['hours_per_week'].apply(work_hours_category)

# Feature 2: Capital Net Gain
data['Capital_Net_Gain'] = df['capital_gain'] - df['capital_loss']

# Log Transformation for skewed features
data['Log_Capital_Gain'] = df['capital_gain'].apply(lambda x: np.log(x + 1))
data['Log_Capital_Loss'] = df['capital_loss'].apply(lambda x: np.log(x + 1))

# Display the first few rows of the modified dataset
print("\nModified Dataset (first 5 rows):")
print(data[['Work_Hours_Category', 'Capital_Net_Gain', 'Log_Capital_Gain', 'Log_Capital_Loss']].head())




Dataset Information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32561 entries, 0 to 32560
Data columns (total 15 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   age             32561 non-null  int64 
 1   workclass       32561 non-null  object
 2   fnlwgt          32561 non-null  int64 
 3   education       32561 non-null  object
 4   education_num   32561 non-null  int64 
 5   marital_status  32561 non-null  object
 6   occupation      32561 non-null  object
 7   relationship    32561 non-null  object
 8   race            32561 non-null  object
 9   sex             32561 non-null  object
 10  capital_gain    32561 non-null  int64 
 11  capital_loss    32561 non-null  int64 
 12  hours_per_week  32561 non-null  int64 
 13  native_country  32561 non-null  object
 14  income          32561 non-null  object
dtypes: int64(6), object(9)
memory usage: 3.7+ MB

Modified Dataset (first 5 rows):
  Work_Hours_Category  Capital_Net_Gai


Task 4: Feature Selection python

In [11]:
# Select numerical columns for outlier detection
numerical_features = ['age', 'fnlwgt', 'education_num', 'capital_gain', 'capital_loss', 'hours_per_week']

# Initialize Isolation Forest
iso_forest = IsolationForest(random_state=42, contamination=0.05)  # 5% contamination
outlier_labels = iso_forest.fit_predict(data[numerical_features])

# Add outlier labels to the dataset (-1 indicates an outlier)
data['outlier'] = outlier_labels

# Count and remove outliers
outliers_count = data[data['outlier'] == -1].shape[0]
data_cleaned = data[data['outlier'] != -1].drop(columns=['outlier'])

print(f"Outliers removed: {outliers_count}")
print(f"Cleaned dataset shape: {data_cleaned.shape}")


Outliers removed: 1628
Cleaned dataset shape: (30933, 15)


In [15]:
# Encode categorical variables and target for mutual information computation
data_encoded = pd.get_dummies(data_cleaned, drop_first=True)
target = data_encoded['income_ >50K']  # Binary encoding of income column

# Compute mutual information scores for all features against the target
mutual_info = mutual_info_classif(data_encoded.drop(columns=['income_ >50K']), target, random_state=42)
mutual_info_series = pd.Series(mutual_info, index=data_encoded.drop(columns=['income_ >50K']).columns)

# Normalize mutual information scores to range [0, 1]
mutual_info_normalized = mutual_info_series / mutual_info_series.max()

# Display mutual information scores sorted
print("Mutual Information Scores (Normalized):")
print(mutual_info_normalized.sort_values(ascending=False))


Mutual Information Scores (Normalized):
marital_status_ Married-civ-spouse    1.000000
capital_gain                          0.673748
marital_status_ Never-married         0.618642
age                                   0.597171
education_num                         0.575861
                                        ...   
occupation_ Armed-Forces              0.000000
native_country_ Honduras              0.000000
workclass_ Self-emp-not-inc           0.000000
native_country_ Iran                  0.000000
native_country_ India                 0.000000
Length: 99, dtype: float64


In [16]:
# Compute correlation matrix for numerical features
correlation_matrix = data_cleaned[numerical_features].corr()

print("Correlation Matrix:")
print(correlation_matrix)


Correlation Matrix:
                     age    fnlwgt  education_num  capital_gain  capital_loss  \
age             1.000000 -0.079026       0.035963      0.093410      0.029835   
fnlwgt         -0.079026  1.000000      -0.039355     -0.012993     -0.011045   
education_num   0.035963 -0.039355       1.000000      0.130080      0.045779   
capital_gain    0.093410 -0.012993       0.130080      1.000000     -0.032021   
capital_loss    0.029835 -0.011045       0.045779     -0.032021      1.000000   
hours_per_week  0.094233 -0.021354       0.134755      0.082962      0.010101   

                hours_per_week  
age                   0.094233  
fnlwgt               -0.021354  
education_num         0.134755  
capital_gain          0.082962  
capital_loss          0.010101  
hours_per_week        1.000000  


In [17]:
# Display mutual information and correlation with the target ('income_ >50K')
target_correlation = data_encoded.corr()['income_ >50K'].sort_values(ascending=False)

print("Correlation with Target (income):")
print(target_correlation)

print("Comparison:")
comparison = pd.DataFrame({
    'Mutual Information': mutual_info_normalized,
    'Correlation': target_correlation.reindex(mutual_info_normalized.index, fill_value=0)
})
print(comparison)


Correlation with Target (income):
income_ >50K                          1.000000
marital_status_ Married-civ-spouse    0.440773
capital_gain                          0.314940
education_num                         0.311494
age                                   0.229483
                                        ...   
relationship_ Unmarried              -0.141777
occupation_ Other-service            -0.150170
relationship_ Not-in-family          -0.186065
relationship_ Own-child              -0.221648
marital_status_ Never-married        -0.312585
Name: income_ >50K, Length: 100, dtype: float64
Comparison:
                                 Mutual Information  Correlation
age                                        0.597171     0.229483
fnlwgt                                     0.278425    -0.012775
education_num                              0.575861     0.311494
capital_gain                               0.673748     0.314940
capital_loss                               0.134059     0.085947